# Wilson-Cowan modelling of binocular rivalry
Computational Neuroscience coursework, 2026.

Original analysis code with outputs cleared and a relative data path. See the README for data requirements and known reproducibility issues, including two missing helper functions.

In [ ]:
# Import the libraries that will be used later

import os
from pathlib import Path
import pandas as pd
import numpy as np

# The data from Han et al. (2024) contains various folders and is complicated
# There are 2 experiments, each experiment contains both a motion and a static task, and some participants even had both 
# practice and actual trials. 
# These steps are to combine them, making it easier for further analysis 

ROOT = Path("../data/Data") 

experiments = ["exp 1", "exp 2"]
conditions = ["Motion", "Static"]

all_data = []
file_log = []

for experiment in experiments:
    for condition in conditions:
        condition_folder = ROOT / experiment / condition

  
        if not condition_folder.exists():
            continue

        participants = sorted(
            [p for p in condition_folder.iterdir() if p.is_dir() and not p.name.startswith(".")]
        )

        for participant_folder in participants:
            participant = participant_folder.name

            txt_files = sorted(participant_folder.rglob("*.txt"))

           
            txt_files = [f for f in txt_files if f.is_file()]


            if len(txt_files) == 0:
                continue


            data_file = txt_files[0]


            df = pd.read_csv(
                data_file,
                sep=None,
                engine="python",
                header=None
            )

      
            df["experiment"] = experiment
            df["condition"] = condition
            df["participant"] = participant
            df["source_file"] = data_file.name
            df["source_path"] = str(data_file)


            all_data.append(df)

  
            file_log.append({
                "experiment": experiment,
                "condition": condition,
                "participant": participant,
                "source_file": data_file.name,
                "source_path": str(data_file),
                "n_rows": df.shape[0],
                "n_cols": df.shape[1]
            })

# Combine files together 
data = pd.concat(all_data, ignore_index=True)

# Create log dataframe
file_log = pd.DataFrame(file_log)

# Basic outputs 
print("Combined data shape:", data.shape)
print("Number of loaded files:", len(file_log))
print("\nFirst 5 loaded files:")
print(file_log.head())

print("\nColumns in combined dataframe:")
print(data.columns.tolist())


# Now the data is successfully combined and loaded 


In [ ]:
# Inspect what was being loaded earlier
print("\nPreview of combined data:")
print(data.head(10))

print("\nLast rows:")
print(data.tail(10))

print("\nDataframe info:")
print(data.info())

print("\nRows per experiment and condition:")
print(
    file_log.groupby(["experiment", "condition"])
    .size()
    .reset_index(name="n_files")
)

print("\nParticipants per experiment and condition:")
print(
    file_log.groupby(["experiment", "condition"])["participant"]
    .nunique()
    .reset_index(name="n_participants")
)

In [ ]:
# Check raw colunms

raw_cols = [c for c in data.columns if isinstance(c, int)]

print("Raw numeric columns:")
print(raw_cols)

print("\nFirst 20 rows of raw columns:")
print(data.loc[:19, raw_cols])

for col in raw_cols:
    print(f"\nColumn {col}")
    print("dtype:", data[col].dtype)
    print("non-null:", data[col].notna().sum())
    print("first 15 unique values:")
    print(pd.Series(data[col]).dropna().unique()[:15])

In [ ]:
# As the columns are checked, move on to data cleaning 
# First, rename the key columns for better understanding 
data = data.rename(columns={
    3: "percept",
    5: "time",
    6: "duration_raw"
})

# Keep only necessary columns for analysis
data = data[
    ["experiment", "condition", "participant", "percept", "time", "duration_raw"]
].copy()

print(data.head())

In [ ]:
# Compute for duration 
# For data in exp 1, use duration_raw
# For data in exp 2, compute from time differences

data = data.sort_values(by=["experiment", "condition", "participant", "time"]).copy()

data["duration_computed"] = (
    data.groupby(["experiment", "condition", "participant"])["time"]
    .diff()
)

# Combine:
data["duration"] = data["duration_raw"].combine_first(data["duration_computed"])

print(data.head(10))

In [ ]:
# Clean the data by removing negative durations and extreme values 

# Remove rows with missing durations 
data = data.dropna(subset=["duration"])

# Remove durations that are invalid 
data = data[data["duration"] > 0]

# Remove extreme values (outliers) 
data = data[data["duration"] < 100]

print("Cleaned data shape:", data.shape)

In [ ]:
# Double check on the percepts 
print(data["percept"].unique())

In [ ]:
# The result from previous cell shows [0 1 2 3]
# 0 means no response or invalid period
# 1 means percept A
# 2 means percep B 
# 3 means mixed percept
# Since binocular rivalry dynamics is the focus of this study, only the stable perception states will be retained (i.e., 1 and 2) 
# 0 and 3 will be excluded 
# The data will now being filtered 

data = data[data["percept"].isin([1, 2])]


In [ ]:
# Turn the raw data into behaviour measure by summarizing the participant data
# This is the participant-level summary 

real_summary = (
    data.groupby(["experiment", "condition", "participant"])
    .agg(
        mean_duration=("duration", "mean"),
        sd_duration=("duration", "std"),
        n_episodes=("duration", "count")
    )
    .reset_index()
)

# Estimate number of switches (episodes - 1)
real_summary["n_switches"] = real_summary["n_episodes"] - 1

print(real_summary.head())
print("\nShape:", real_summary.shape)


# Each row represents an individual participant in one condition 
# mean_duratoin reflects the average time of a percept remains dominant
# n_episodes is the approximation of perceptual alternations 
# n_switches = episodes - 1 

In [ ]:
# Looking at the conditions 
# This is the contiion-level summary 
condition_summary = (
    real_summary.groupby(["experiment", "condition"])
    .agg(
        mean_duration=("mean_duration", "mean"),
        sd_duration=("mean_duration", "std"),
        mean_switches=("n_switches", "mean"),
        sd_switches=("n_switches", "std"),
        n_participants=("participant", "count")
    )
    .reset_index()
)

print(condition_summary)

In [ ]:
# Plot results for dominance duration 
import matplotlib.pyplot as plt

for exp in real_summary["experiment"].unique():
    subset = real_summary[real_summary["experiment"] == exp]

    plt.figure()
    subset.groupby("condition")["mean_duration"].mean().plot(kind="bar")
    plt.ylabel("Mean dominance duration")
    plt.title(f"Dominance duration ({exp})")
    plt.xticks(rotation=0)
    plt.show()

In [ ]:
# Plot results for switches
for exp in real_summary["experiment"].unique():
    subset = real_summary[real_summary["experiment"] == exp]

    plt.figure()
    subset.groupby("condition")["n_switches"].mean().plot(kind="bar")
    plt.ylabel("Mean number of switches")
    plt.title(f"Switches ({exp})")
    plt.xticks(rotation=0)
    plt.show()

In [ ]:
# Run the statisticl testings 
# Run an independent sample t-test comparing Motion and Static conditions for exp 1 and exp 2
# Missing values will be ommited with nan_policy="omit"
from scipy.stats import ttest_ind

for exp in real_summary["experiment"].unique():
    subset = real_summary[real_summary["experiment"] == exp]

    motion = subset[subset["condition"] == "Motion"]
    static = subset[subset["condition"] == "Static"]

    t_dur, p_dur = ttest_ind(motion["mean_duration"], static["mean_duration"], nan_policy="omit")
    t_sw, p_sw = ttest_ind(motion["n_switches"], static["n_switches"], nan_policy="omit")

    print(f"\n{exp}")
    print(f"Duration: t = {t_dur:.3f}, p = {p_dur:.3f}")
    print(f"Switches: t = {t_sw:.3f}, p = {p_sw:.3f}")

In [ ]:
# Since Han et al looks at if Binocular rivalry (BR) can be generalized into real world, we should also 
# compare performances between exp 1 (which used implausible and semi-implausible cues) and exp 2 (which used plausible cues) 
# if they do not differ significantly, this suggests that BR is not just a laboratory phenomenon  

from scipy.stats import ttest_ind

# Compare mean duration across experiments
exp1 = real_summary[real_summary["experiment"] == "exp 1"]
exp2 = real_summary[real_summary["experiment"] == "exp 2"]

# Duration comparison
t_dur, p_dur = ttest_ind(exp1["mean_duration"], exp2["mean_duration"], nan_policy="omit")

# Switch comparison
t_sw, p_sw = ttest_ind(exp1["n_switches"], exp2["n_switches"], nan_policy="omit")

print("Comparison between experiments (exp 1 vs exp 2)")
print(f"Duration: t = {t_dur:.3f}, p = {p_dur:.3f}")
print(f"Switches: t = {t_sw:.3f}, p = {p_sw:.3f}")


# plot the results
plt.figure()
real_summary.groupby("experiment")["mean_duration"].mean().plot(kind="bar")
plt.ylabel("Mean dominance duration")
plt.title("Dominance duration: exp 1 vs exp 2")
plt.xticks(rotation=0)
plt.show()

plt.figure()
real_summary.groupby("experiment")["n_switches"].mean().plot(kind="bar")
plt.ylabel("Mean switches")
plt.title("Switches: exp 1 vs exp 2")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Use Wilson-Cowan model to stimulate the data
# First set up for the model 


# Reproducibility
np.random.seed(42)

# Time settings
dt = 0.1
T = 300
time = np.arange(0, T, dt)

# Sigmoid activation function
# Parameters: 
# x: float or np.ndarray (input to the population)
# a: float, the stepness of the sigmoid 
# theta: also float, the activation threashold 

def sigmoid(x, a=4.0, theta=0.5):
    return 1 / (1 + np.exp(-a * (x - theta)))


In [ ]:
# Stimulate the activities of two populations 

# Parameters: 
# w_self: self-excitation stregth in each population 
# w_inhib: mutual inhibition strength between populations 
# I: baseline external input equally applied to both populations 
# noise_level: standard deviation of Gaussian noise added at each time step 
# beta: strength of adaptation (fatigue) 
# tua_a: time constant of adaptation 

def run_wc_model(w_self=1.0, w_inhib=1.0, I=-0.5, noise_level=0.05, beta=2.0, tau_a=5.0):
    # Initialise activity of the two competing populations
    E1 = np.zeros(len(time))
    E2 = np.zeros(len(time))

    # Adaptation variables
    A1 = np.zeros(len(time))
    A2 = np.zeros(len(time))

    # Initial conditions
    E1[0] = 0.2
    E2[0] = 0.1

    # Simulate dynamics using Euler integration
    for t in range(len(time) - 1):

        # Add Gaussian noise to each population
        noise1 = np.random.normal(0, noise_level)
        noise2 = np.random.normal(0, noise_level)

        # Wilson–Cowan equations with self-excitation, mutual inhibition, and adaptation
        dE1 = -E1[t] + sigmoid(w_self * E1[t] - w_inhib * E2[t] - beta * A1[t] + I + noise1)
        dE2 = -E2[t] + sigmoid(w_self * E2[t] - w_inhib * E1[t] - beta * A2[t] + I + noise2)

        # Adaptation increases with activity and decays gradually over time
        dA1 = (-A1[t] + E1[t]) / tau_a
        dA2 = (-A2[t] + E2[t]) / tau_a

        # Euler update
        E1[t + 1] = E1[t] + dt * dE1
        E2[t + 1] = E2[t] + dt * dE2
        A1[t + 1] = A1[t] + dt * dA1
        A2[t + 1] = A2[t] + dt * dA2

    return E1, E2

In [ ]:
# Run the baseline stimulation (without any parameters altered) 
E1, E2 = run_wc_model(
    w_self=1.0,
    w_inhib=1.0,
    I=-0.5,
    noise_level=0.05,
    beta=2.0,
    tau_a=5.0
)

plt.figure(figsize=(10, 4))
plt.plot(time, E1, label="Population 1")
plt.plot(time, E2, label="Population 2")
plt.xlabel("Time")
plt.ylabel("Activity")
plt.title("Baseline Wilson-Cowan rivalry simulation")
plt.legend()
plt.show()

In [ ]:
# Extract percepts and behaviour measures for stimulated data
percepts = extract_percepts(E1, E2)
n_switches, mean_duration, durations = compute_model_measures(percepts, dt)

print("Baseline model results")
print("Number of switches:", n_switches)
print("Mean dominance duration:", mean_duration)


In [ ]:
# The previous baseline results showed that switches are too often 
# Now test if changing parameters can slow this down 
# Sweep w_inhib 

inhibition_values = np.linspace(0.5, 2.5, 9)

model_results = []

for w_inhib in inhibition_values:
    E1, E2 = run_wc_model(
        w_self=1.0,
        w_inhib=w_inhib,
        I=-0.5,
        noise_level=0.05,
        beta=2.0,
        tau_a=5.0
    )
    
    percepts = extract_percepts(E1, E2)
    n_switches, mean_duration, durations = compute_model_measures(percepts, dt)
    
    model_results.append({
        "w_inhib": w_inhib,
        "n_switches": n_switches,
        "mean_duration": mean_duration
    })

model_results = pd.DataFrame(model_results)
print(model_results)


# plot it too 
plt.figure()
plt.plot(model_results["w_inhib"], model_results["n_switches"], marker="o")
plt.xlabel("Mutual inhibition strength")
plt.ylabel("Number of switches")
plt.title("Effect of inhibition on switching")
plt.show()

plt.figure()
plt.plot(model_results["w_inhib"], model_results["mean_duration"], marker="o")
plt.xlabel("Mutual inhibition strength")
plt.ylabel("Mean dominance duration")
plt.title("Effect of inhibition on dominance duration")
plt.show()

In [ ]:
# The previous results suggest that altering inhibition strength does not affect switching dynamics
# Now sweep beta instead 
beta_values = np.linspace(0.5, 4.0, 10)

model_results_beta = []

for beta in beta_values:
    E1, E2 = run_wc_model(
        w_self=1.0,
        w_inhib=1.0,
        I=-0.5,
        noise_level=0.05,
        beta=beta,
        tau_a=5.0
    )
    
    percepts = extract_percepts(E1, E2)
    n_switches, mean_duration, durations = compute_model_measures(percepts, dt)
    
    model_results_beta.append({
        "beta": beta,
        "n_switches": n_switches,
        "mean_duration": mean_duration
    })

model_results_beta = pd.DataFrame(model_results_beta)
print(model_results_beta)

# Plot it

plt.figure()
plt.plot(model_results_beta["beta"], model_results_beta["n_switches"], marker="o")
plt.xlabel("Adaptation strength (beta)")
plt.ylabel("Number of switches")
plt.title("Effect of adaptation on switching")
plt.show()

plt.figure()
plt.plot(model_results_beta["beta"], model_results_beta["mean_duration"], marker="o")
plt.xlabel("Adaptation strength (beta)")
plt.ylabel("Mean dominance duration")
plt.title("Effect of adaptation on dominance duration")
plt.show()


In [ ]:
# Print model summary
print("Model results (baseline):")
print(f"Switches: {n_switches}")
print(f"Mean duration: {mean_duration:.3f}")

In [ ]:
# Comparing data from Han et al (2024) with the stimulated data
print("Real vs Model (rough comparison)")
print("Real mean duration:", real_summary["mean_duration"].mean())
print("Model mean duration:", mean_duration)